# Code 10.7.1: The actor-critic networks

Separate policy and value networks with two hidden layers of 64 tanh units, orthogonal initialization, and a small initial scale for the policy's output so that the initial policy is nearly uniform.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

def layer(inp, out, std=np.sqrt(2)):
    """A linear layer with orthogonal weights and zero bias."""
    lin = nn.Linear(inp, out)
    nn.init.orthogonal_(lin.weight, std)
    nn.init.zeros_(lin.bias)
    return lin

class ActorCritic(nn.Module):
    """Separate policy (actor) and value (critic) networks."""
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.actor = nn.Sequential(layer(obs_dim, hidden), nn.Tanh(),
                                   layer(hidden, hidden), nn.Tanh(),
                                   layer(hidden, n_actions, std=0.01))
        self.critic = nn.Sequential(layer(obs_dim, hidden), nn.Tanh(),
                                    layer(hidden, hidden), nn.Tanh(),
                                    layer(hidden, 1, std=1.0))

    def dist(self, s):
        return torch.distributions.Categorical(logits=self.actor(s))

    def value(self, s):
        return self.critic(s).squeeze(-1)
